# Self-Attention, Cross-Attention, and Future Leakage Audit
Pure NumPy implementation.

In [1]:
import numpy as np

X = np.array([[1., 0., 1., 0.],
              [0., 1., 1., 0.],
              [1., 1., 0., 1.]])

ctx = np.array([[2., 0., 0., 1.],
                [0., 2., 1., 0.],
                [1., 1., 1., 1.],
                [0., 0., 2., 2.],
                [1., 0., 1., 0.]])

In [2]:
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z_max = np.max(z, axis=axis, keepdims=True)
    exp_z = np.exp(z - z_max)
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

In [3]:
def causal_mask(n):
    mask = np.zeros((n, n), dtype=float)
    mask[np.triu_indices(n, k=1)] = -np.inf
    return mask

In [4]:
def self_attention(X, mask=None):
    d = X.shape[-1]
    scores = X @ X.T / np.sqrt(d)

    if mask is not None:
        scores = scores + mask

    weights = softmax(scores, axis=-1)
    out = weights @ X
    return out, weights

In [5]:
def cross_attention(X, context):
    d = X.shape[-1]
    scores = X @ context.T / np.sqrt(d)
    weights = softmax(scores, axis=-1)
    out = weights @ context
    return out, weights

In [6]:
def audit_future_leakage(weights):
    weights = np.asarray(weights)
    if weights.ndim != 2 or weights.shape[0] != weights.shape[1]:
        return False
    return bool(np.any(np.triu(weights, k=1) > 0))

In [7]:
out, w_full = self_attention(X)
out, w_causal = self_attention(X, mask=causal_mask(3))

print(audit_future_leakage(w_causal))
print(audit_future_leakage(w_full))

out, w = cross_attention(X, ctx)
print(w.shape)

False
True
(3, 5)
